In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%pip install --upgrade torch-geometric-signed-directed networkx

  Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl.metadata (27 kB)
  Using cached torch_geometric-2.8.0.post1-py3-none-any.whl.metadata (64 kB)
  Using cached xxhash-4.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (17 kB)
Using cached torch_geometric_signed_directed-1.1.1-py3-none-any.whl (119 kB)
Using cached torch_geometric-2.8.0.post1-py3-none-any.whl (1.3 MB)
Using cached xxhash-4.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (268 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch-geometric-signed-directed]ic]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# dbutils.library.restartPython()

In [4]:
import os

# ============================================================
# Parameters
# ============================================================
N_USERS = 150  # Total users to sample (proportionally across groups). None = all users.
EXPERIMENT_TAG = "v3_full"  # Experiment identifier
RESET_GNN_TRAINING = False  # Set True to clear checkpoints/logs and restart GNN training from scratch

# Paths
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
# DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
DATA_PATH = "/serafin/pcelayes/repos/sna_classifier/data"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}" if N_USERS else EXPERIMENT_TAG
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
# Training hyperparameters
TRAIN_CHUNK_SIZE = 10  # Users per chunk for on-the-fly GNN sample generation during training
# EPOCHS = 60  # Total training epochs.
# LOG_EVERY_N_STEPS = 300  # Log metrics (loss, val F1) every N training steps.
# TRAIN_F1_EVERY_N_EPOCHS = 5  # Compute train F1 every N epochs. None = skip during training (always computed at end on best model).

EPOCHS = 4  # Total training epochs.
LOG_EVERY_N_STEPS = 100  # Log metrics (loss, val F1) every N training steps.
TRAIN_F1_EVERY_N_EPOCHS = 2  # Compute train F1 every N epochs. None = skip during training (always computed at end on best model).

PATIENCE = 20  # Early stopping: checkpoints without val F1 improvement. None = disabled.
GRADIENT_ACCUMULATION_STEPS = 4  # None or 1 to disable. Effective batch = batch_size * this.
MIXED_PRECISION = True  # Use float16 autocast + GradScaler for faster CUDA training.
MAX_VAL_SAMPLES = 10_000  # Cap validation samples (randomly sampled from test set). None = use all.

# Pre-trained weights initialization (set to a .pt file path to warm-start from another run)
# Example: "./experiments/v3_full_N150/best_retweet_gnn_general.pt"
INIT_WEIGHTS_PATH = "./experiments/v2_N20/best_retweet_gnn_general.pt"  # None = train from scratch
FINETUNE_LR_FACTOR = 0.1  # When fine-tuning (INIT_WEIGHTS_PATH set), multiply base LR by this factor
FINETUNE_WD_FACTOR = 0.2  # When fine-tuning, multiply base weight_decay by this factor (less L2 → preserve pre-trained structure)
FINETUNE_WARMUP_EPOCHS = 2  # Shorter warmup when fine-tuning (weights already in a good region)

print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")
if INIT_WEIGHTS_PATH:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
else:
    print("Training from scratch.")

Experiment: v3_full_N150
Output dir: ./experiments/v3_full_N150
Loading pre-trained weights from: ./experiments/v2_N20/best_retweet_gnn_general.pt


In [5]:
import shutil

deleted = []

# --- RESET_GNN_TRAINING: clear checkpoints and logs (forces training restart) ---
if RESET_GNN_TRAINING:
    print("⚠️  RESET_GNN_TRAINING=True — clearing checkpoints and training logs...")
    _training_artifacts = [
        "best_retweet_gnn_general.pt",
        "training.log",
        "training_history.json",
        "checkpoint.pt",
    ]
    for fname in _training_artifacts:
        fpath = os.path.join(EXPERIMENT_DIR, fname)
        if os.path.exists(fpath):
            os.remove(fpath)
            deleted.append(fpath)

    print(f"  Cleared training artifacts. GNN training will start fresh.")
else:
    print("RESET_GNN_TRAINING=False — resuming from existing checkpoint if available.")

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

if deleted:
    print(f"\nTotal deleted items: {len(deleted)}")
    for d in deleted:
        print(f"  - {d}")

RESET_GNN_TRAINING=False — resuming from existing checkpoint if available.


In [6]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow C++ info/warning logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # Suppress oneDNN messages

import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

sys.path.insert(0, str(sources_path))

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

In [7]:
import torch
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {props.name}, free={free/1e9:.1f}GB / {total/1e9:.1f}GB")

GPU 0: NVIDIA A30, free=19.1GB / 19.3GB


In [8]:
import subprocess

def get_free_memory_per_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    return [int(x) for x in result.stdout.strip().split("\n")]

def get_best_gpu():
    free_mem = get_free_memory_per_gpu()
    best_gpu = max(range(len(free_mem)), key=lambda i: free_mem[i])
    print(f"Free memory per GPU (MiB): {free_mem}")
    print(f"Selected GPU {best_gpu} with {free_mem[best_gpu]} MiB free")
    return best_gpu

device = torch.device(f"cuda:{get_best_gpu()}" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Free memory per GPU (MiB): [18208]
Selected GPU 0 with 18208 MiB free


In [7]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

Graph loaded: 5589 nodes, 261005 edges


In [9]:
# Load user splits
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

print(f"Groups in user_splits: {list(user_splits.keys())}")
for k, v in user_splits.items():
    print(f"  {k}: {len(v)} users")

# Define train groups vs test groups
TRAIN_GROUPS = ["u_train", "au_train"]
TEST_GROUPS = [g for g in user_splits.keys() if g not in TRAIN_GROUPS]
print(f"\nTrain groups: {TRAIN_GROUPS}")
print(f"Test groups: {TEST_GROUPS}")

# ---------------------------------------------------------------
# Deterministic sample tied to FINAL_TAG: save/load sampled user IDs
# so that re-runs for the same experiment tag use the exact same users.
# ---------------------------------------------------------------
SAMPLE_PATH = f"{EXPERIMENT_DIR}/user_sample.json"

if os.path.exists(SAMPLE_PATH):
    # --- FAST PATH: load previously saved sample for this tag ---
    with open(SAMPLE_PATH) as f:
        saved_sample = json.load(f)  # {group: [uid, uid, ...]}
    print(f"\nLoading saved user sample from {SAMPLE_PATH}")

    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []
    for group, uids in saved_sample.items():
        user_data[group] = {}
        for uid in uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data on reload"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))

    print(f"  Loaded users per group:")
    for group in saved_sample:
        print(f"    {group}: {len(user_data[group])}/{len(saved_sample[group])}")
    print(f"  Total: {sum(len(user_data[g]) for g in user_data)}")
    if failed_users:
        print(f"  Failed on reload: {len(failed_users)}")

else:
    # --- FIRST RUN: load all users, sample, then save ---
    print(f"\nNo saved sample for {FINAL_TAG}, loading all users...")
    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []

    for group in user_splits:
        user_data[group] = {}
        group_uids = user_splits[group]
        for uid in group_uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))
                continue

    print(f"\nLoaded users per group:")
    total_valid = 0
    for group in user_splits:
        n = len(user_data[group])
        total_valid += n
        print(f"  {group}: {n}/{len(user_splits[group])} valid")
    print(f"  Total valid: {total_valid}")
    print(f"  Failed: {len(failed_users)}")

    # Proportional sampling if N_USERS is set
    if N_USERS is not None:
        group_sizes = {g: len(user_data[g]) for g in user_splits}
        total_available = sum(group_sizes.values())

        # Step 1: Keep proportions between train and test groups
        train_available = sum(group_sizes.get(g, 0) for g in TRAIN_GROUPS)
        test_available = sum(group_sizes.get(g, 0) for g in TEST_GROUPS)
        train_slots = int(round(N_USERS * train_available / total_available))
        test_slots = N_USERS - train_slots

        # Step 2: Train groups — prioritize u_train first, fill remainder with au_train
        n_u_train = min(group_sizes.get("u_train", 0), train_slots)
        n_au_train = min(group_sizes.get("au_train", 0), train_slots - n_u_train)
        raw_alloc = {"u_train": n_u_train, "au_train": n_au_train}

        # Step 3: Test groups — proportional allocation within test slots
        test_group_sizes = {g: group_sizes.get(g, 0) for g in TEST_GROUPS if group_sizes.get(g, 0) > 0}
        total_test_available = sum(test_group_sizes.values())
        for g in TEST_GROUPS:
            if total_test_available > 0 and group_sizes.get(g, 0) > 0:
                raw_alloc[g] = int(round(test_slots * group_sizes[g] / total_test_available))
            else:
                raw_alloc[g] = 0
        # Adjust test rounding to hit exact test_slots
        test_diff = test_slots - sum(raw_alloc.get(g, 0) for g in TEST_GROUPS)
        for g in sorted(TEST_GROUPS, key=lambda g: group_sizes.get(g, 0), reverse=True):
            if test_diff == 0:
                break
            adjustment = 1 if test_diff > 0 else -1
            raw_alloc[g] = max(1, raw_alloc[g] + adjustment)
            test_diff -= adjustment

        # Sample from each group
        sampled_user_data = {}
        for g in user_splits:
            if g not in raw_alloc or raw_alloc[g] == 0:
                sampled_user_data[g] = {}
                continue
            uids = list(user_data[g].keys())
            n_sample = min(raw_alloc[g], len(uids))
            sampled_uids = sample(uids, n_sample)
            sampled_user_data[g] = {uid: user_data[g][uid] for uid in sampled_uids}
        user_data = sampled_user_data

        print(f"\nSampled {N_USERS} users (train priority: u_train first, then au_train):")
        for g in user_splits:
            print(f"  {g}: {len(user_data[g])} (target {raw_alloc.get(g, 0)})")
        print(f"  Total sampled: {sum(len(user_data[g]) for g in user_splits)}")

    # Save the sample (user IDs per group) for reproducibility
    sample_to_save = {g: list(user_data[g].keys()) for g in user_data}
    with open(SAMPLE_PATH, "w") as f:
        json.dump(sample_to_save, f, indent=2)
    print(f"  Saved user sample to {SAMPLE_PATH}")

# Flat list of train-group users (for GNN training)
valid_users = list(user_data.get("u_train", {}).keys()) + list(user_data.get("au_train", {}).keys())
# Baseline SVC uses only u_train users
baseline_users = list(user_data.get("u_train", {}).keys())
print(f"\nTrain-group valid users: {len(valid_users)} (baseline SVC: {len(baseline_users)} from u_train only)")

Groups in user_splits: ['u_train', 'u_test', 'au_train', 'au_test']
  u_train: 97 users
  u_test: 97 users
  au_train: 3152 users
  au_test: 1352 users

Train groups: ['u_train', 'au_train']
Test groups: ['u_test', 'au_test']

Loading saved user sample from ./experiments/v3_full_N150/user_sample.json
  Loaded users per group:
    u_train: 96/96
    u_test: 5/5
    au_train: 12/12
    au_test: 37/37
  Total: 150

Train-group valid users: 108 (baseline SVC: 96 from u_train only)


## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [ ]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

# ---------------------------------------------------------------------------
# User-level baseline cache (shared across experiments)
# Each user's result is stored independently so we never recompute a user.
# ---------------------------------------------------------------------------
BASELINE_USER_CACHE_PATH = f"{DATA_PATH}/baseline_svc_user_cache.pkl"

# Initialize cache from v3_full_N150 if it doesn't exist yet
if not os.path.exists(BASELINE_USER_CACHE_PATH):
    _init_path = "./experiments/v3_full_N150/baseline_svc_results.pkl"
    if os.path.exists(_init_path):
        print(f"Initializing user-level baseline cache from {_init_path}...")
        with open(_init_path, "rb") as f:
            _init_data = pickle.load(f)
        _cache = {}
        _init_f1s = _init_data["baseline_f1s"]
        _init_params = _init_data["baseline_best_params"]
        _init_preds = _init_data["all_baseline_test_preds"]
        # all_baseline_test_preds is ordered parallel to baseline_f1s keys
        _user_ids = list(_init_f1s.keys())
        for idx, uid in enumerate(_user_ids):
            preds, labels = _init_preds[idx]
            _cache[uid] = {
                "f1": _init_f1s[uid],
                "best_params": _init_params[uid],
                "preds": preds,
                "labels": labels,
            }
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump(_cache, f)
        print(f"  Initialized cache with {len(_cache)} users from v3_full_N150.")
        del _init_data, _cache, _init_f1s, _init_params, _init_preds
    else:
        print(f"No v3_full_N150 results found at {_init_path}, starting empty cache.")
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump({}, f)

# Load existing user-level cache
with open(BASELINE_USER_CACHE_PATH, "rb") as f:
    baseline_user_cache = pickle.load(f)
print(f"Baseline user cache: {len(baseline_user_cache)} users already computed.")

# Determine which baseline users still need processing
users_to_compute = [uid for uid in baseline_users if uid not in baseline_user_cache]
users_cached = [uid for uid in baseline_users if uid in baseline_user_cache]
print(f"  This experiment: {len(baseline_users)} baseline users")
print(f"  Already cached:  {len(users_cached)}")
print(f"  Need computing:  {len(users_to_compute)}")

if users_to_compute:
    # Reduced hyperparameter grid for faster iteration
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    t0 = time.time()
    for i, uid in enumerate(users_to_compute):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data["u_train"][uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        # Save to cache immediately
        baseline_user_cache[uid] = {
            "f1": best_f1,
            "best_params": best_params,
            "preds": best_preds,
            "labels": np.array(y_te),
        }

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(users_to_compute) - i - 1)
        print(f"  [{i+1:>3}/{len(users_to_compute)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {int(elapsed)//60}m{int(elapsed)%60:02d}s | ETA {int(remaining)//60}m{int(remaining)%60:02d}s)")

    total_time = time.time() - t0
    print(f"\nComputed {len(users_to_compute)} new users in {total_time:.1f}s "
          f"({total_time/len(users_to_compute):.1f}s/user avg).")

    # Persist updated cache
    with open(BASELINE_USER_CACHE_PATH, "wb") as f:
        pickle.dump(baseline_user_cache, f)
    print(f"  Cache updated: {len(baseline_user_cache)} total users.")
else:
    print("All baseline users already cached — no computation needed.")

# --- Assemble experiment-level results from cache ---
baseline_f1s = {uid: baseline_user_cache[uid]["f1"] for uid in baseline_users}
baseline_best_params = {uid: baseline_user_cache[uid]["best_params"] for uid in baseline_users}
all_baseline_test_preds = [
    (baseline_user_cache[uid]["preds"], baseline_user_cache[uid]["labels"])
    for uid in baseline_users
]

# Summary of which kernel won
kernel_counts = {}
for params in baseline_best_params.values():
    k = params[0]
    kernel_counts[k] = kernel_counts.get(k, 0) + 1
print(f"\nBaseline results for {len(baseline_users)} users — kernel distribution: {kernel_counts}")

In [ ]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

GNN samples are created **on-the-fly** from user dataframes (no caching/storage of samples):
- **Training**: iterates users in shuffled chunks of `TRAIN_CHUNK_SIZE`, transforms to GNN samples,
  yields shuffled batches within each chunk, frees memory after each chunk.
- **Validation**: pre-computes samples (capped at `MAX_VAL_SAMPLES`, proportional across users),
  cached in CPU memory for fast repeated evaluation.

Architecture from 2.0, with aggressive anti-overfitting (test set has entirely unseen users):
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `dropout=0.5, drop_edge_rate=0.1` (stochastic regularization)
- `gate_param init=0.0` (sigmoid=0.5, balanced — shortcut generalizes better to unseen users)
- `weight_decay=1e-3` (L2)
- `lr=3e-3` with 5-epoch linear warmup + cosine decay
- `epochs=60, batch_size=256, log_every_n_steps=300`

In [ ]:
from gnn_models import (
    PretrainedEmbeddingLookup, RetweetGNN,
    soft_f1_loss, combined_loss, evaluate, train_model,
)

In [ ]:
# ---------------------------------------------------------------------------
# Build user assignments for training and validation.
# GNN samples are created on-the-fly by the loaders (no caching).
# ---------------------------------------------------------------------------
BATCH_SIZE = 256

# Train: all users from TRAIN_GROUPS (their "train" split)
train_user_items = []
for group in TRAIN_GROUPS:
    for uid in user_data.get(group, {}):
        train_user_items.append((group, uid))

# Val: train-group users' "test" split + test-group users' both splits
val_user_splits = []
for group in TRAIN_GROUPS:
    for uid in user_data.get(group, {}):
        val_user_splits.append((group, uid, "test"))
for group in TEST_GROUPS:
    for uid in user_data.get(group, {}):
        val_user_splits.append((group, uid, "train"))
        val_user_splits.append((group, uid, "test"))

print(f"Train users: {len(train_user_items)} (from {TRAIN_GROUPS})")
print(f"Val user/split combos: {len(val_user_splits)} "
      f"(from {TRAIN_GROUPS} test + {TEST_GROUPS} both)")
print(f"\nGNN samples will be created on-the-fly (chunk_size={TRAIN_CHUNK_SIZE}).")
print(f"Val samples capped at {MAX_VAL_SAMPLES} (proportional across users).")

In [ ]:
import gc
from random import shuffle as _shuffle_list
from torch_geometric.data import Data, Batch


# ---------------------------------------------------------------------------
# Helper: convert a raw GNN sample dict to a PyG Data object (CPU tensors)
# ---------------------------------------------------------------------------
def _sample_to_pyg_data(sample):
    """Convert a raw sample dict from create_gnn_train_val_samples to PyG Data."""
    central_id = int(sample["central_user_id"])
    neighbor_ids = (
        sample["neighbor_ids"].tolist()
        if hasattr(sample["neighbor_ids"], "tolist")
        else list(sample["neighbor_ids"])
    )
    all_ids = [central_id] + neighbor_ids
    num_nodes = len(all_ids)

    user_ids = torch.tensor(all_ids, dtype=torch.long)

    retweeted_raw = sample["retweeted_ids"]
    retweeted_set = set(
        int(r) for r in (retweeted_raw.tolist() if hasattr(retweeted_raw, "tolist") else retweeted_raw)
    )
    retweet_flag = torch.tensor(
        [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
        dtype=torch.float,
    ).unsqueeze(1)

    ei = sample["edge_index"]
    if hasattr(ei, "__len__") and len(ei) > 0:
        ei_arr = np.array(ei, dtype=np.int64) if not isinstance(ei, np.ndarray) else ei.astype(np.int64)
        edge_index = torch.from_numpy(ei_arr).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    label = torch.tensor(int(sample["label"]), dtype=torch.long)

    return Data(
        user_ids=user_ids,
        retweet_flag=retweet_flag,
        edge_index=edge_index,
        y=label,
        num_nodes=num_nodes,
        central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(
            0, torch.tensor([0]), True
        ),
    )


# ---------------------------------------------------------------------------
# ChunkedGNNTrainLoader: on-the-fly transformation in user chunks
# ---------------------------------------------------------------------------
class ChunkedGNNTrainLoader:
    """Training loader that transforms GNN samples on-the-fly in user chunks.

    Each epoch:
      1. Shuffles the user list
      2. Processes users in chunks of `chunk_size`
      3. For each chunk: transforms all users' training data to GNN samples,
         shuffles the combined samples, yields batches of `batch_size`
      4. Frees chunk samples after processing

    All tensors are kept on CPU. The training loop moves batches to GPU.
    """

    def __init__(self, user_items, user_data, graph, batch_size, chunk_size=10):
        self.user_items = list(user_items)
        self.user_data = user_data
        self.graph = graph
        self.batch_size = batch_size
        self.chunk_size = chunk_size
        self._failed_users = []
        self._estimated_samples = None

    @property
    def total_samples(self):
        if self._estimated_samples is None:
            self._estimated_samples = sum(self.get_label_counts().values())
        return self._estimated_samples

    def __len__(self):
        return self.total_samples // self.batch_size

    def __iter__(self):
        """Yield PyG Batch objects, processing users in chunks."""
        users = list(self.user_items)
        _shuffle_list(users)

        total_yielded = 0
        self._failed_users = []

        for chunk_start in range(0, len(users), self.chunk_size):
            chunk_users = users[chunk_start : chunk_start + self.chunk_size]

            # Transform all users in this chunk to GNN samples
            chunk_data_list = []
            for group, uid in chunk_users:
                X_tr, X_te, y_tr, y_te = self.user_data[group][uid]
                try:
                    train_samples, _ = create_gnn_train_val_samples(
                        uid, self.graph, X_tr, y_tr, X_te, y_te
                    )
                    for s in train_samples:
                        chunk_data_list.append(_sample_to_pyg_data(s))
                except Exception as e:
                    self._failed_users.append((group, uid, str(e)))
                    continue

            if not chunk_data_list:
                continue

            # Shuffle within chunk
            _shuffle_list(chunk_data_list)

            # Yield batches
            for start in range(0, len(chunk_data_list) - self.batch_size + 1, self.batch_size):
                batch = Batch.from_data_list(chunk_data_list[start : start + self.batch_size])
                yield batch
                total_yielded += self.batch_size

            # Free chunk memory
            del chunk_data_list
            gc.collect()

        # Update estimate for __len__ after first full epoch
        if total_yielded > 0:
            self._estimated_samples = total_yielded

    def get_label_counts(self):
        """Count labels directly from y_tr (avoids full GNN transform pass)."""
        counts = {}
        for group, uid in self.user_items:
            _, _, y_tr, _ = self.user_data[group][uid]
            y_arr = np.asarray(y_tr).ravel()
            for label in y_arr:
                counts[int(label)] = counts.get(int(label), 0) + 1
        return counts


# ---------------------------------------------------------------------------
# CachedValLoader: pre-computes and caches val samples in CPU memory
# ---------------------------------------------------------------------------
class CachedValLoader:
    """Validation loader that pre-computes all samples and caches in CPU memory.

    If max_samples is set, subsamples input rows BEFORE GNN transformation
    (proportionally across users) to avoid wasting compute on samples that
    would be discarded.
    """

    def __init__(self, val_user_splits, user_data, graph, batch_size, max_samples=None, cache_path=None):
        self.batch_size = batch_size
        self._data_list = []
        self._failed_users = []

        # Try loading from chunked cache directory
        if cache_path and os.path.isdir(cache_path):
            chunk_files = sorted(f for f in os.listdir(cache_path) if f.startswith("chunk_") and f.endswith(".pt"))
            if chunk_files:
                print(f"Loading cached val samples from: {cache_path} ({len(chunk_files)} chunks)")
                self._data_list = []
                for cf in chunk_files:
                    self._data_list.extend(torch.load(os.path.join(cache_path, cf), map_location="cpu", weights_only=False))
                print(f"  Loaded {len(self._data_list)} cached samples, {len(self)} batches")
                return

        n_total = len(val_user_splits)
        print(f"Pre-computing validation samples ({n_total} user/split combos)...")

        # Count available rows per user/split (cheap — just .shape[0])
        row_counts = []
        for group, uid, split_name in val_user_splits:
            X_tr, X_te, _, _ = user_data[group][uid]
            row_counts.append(X_tr.shape[0] if split_name == "train" else X_te.shape[0])

        total_available_rows = sum(row_counts)

        # Determine per-user/split allocation
        if max_samples and total_available_rows > max_samples:
            sample_rate = max_samples / total_available_rows
            allocations = [max(1, int(n * sample_rate)) for n in row_counts]
            # Trim to budget by reducing largest allocations
            while sum(allocations) > max_samples:
                max_idx = max(range(len(allocations)), key=lambda i: allocations[i])
                allocations[max_idx] -= 1
            do_subsample = True
            print(f"  Sampling enabled: {max_samples} target from {total_available_rows} available rows")
        else:
            allocations = row_counts
            do_subsample = False

        # Transform: subsample rows first, then create GNN samples
        for i, (group, uid, split_name) in enumerate(val_user_splits):
            if (i + 1) % 5 == 0 or (i + 1) == n_total:
                print(f"  [{i+1}/{n_total}] {len(self._data_list)} samples so far...", end="\r")

            X_tr, X_te, y_tr, y_te = user_data[group][uid]
            n_keep = allocations[i]

            try:
                if do_subsample:
                    if split_name == "test":
                        # Subsample test rows; pass 1-row dummy train to skip unused split
                        idx = np.random.choice(X_te.shape[0], size=min(n_keep, X_te.shape[0]), replace=False)
                        _, test_samples = create_gnn_train_val_samples(
                            uid, graph, X_tr.iloc[:1], y_tr.iloc[:1], X_te.iloc[idx], y_te.iloc[idx]
                        )
                        samples = test_samples
                    else:
                        # Subsample train rows; pass 1-row dummy test to skip unused split
                        idx = np.random.choice(X_tr.shape[0], size=min(n_keep, X_tr.shape[0]), replace=False)
                        train_samples, _ = create_gnn_train_val_samples(
                            uid, graph, X_tr.iloc[idx], y_tr.iloc[idx], X_te.iloc[:1], y_te.iloc[:1]
                        )
                        samples = train_samples
                else:
                    train_samples, test_samples = create_gnn_train_val_samples(
                        uid, graph, X_tr, y_tr, X_te, y_te
                    )
                    samples = train_samples if split_name == "train" else test_samples

                for s in samples:
                    self._data_list.append(_sample_to_pyg_data(s))
            except Exception as e:
                self._failed_users.append((group, uid, str(e)))
                continue
        print()  # newline after \r progress

        gc.collect()

        print(
            f"  Cached {len(self._data_list)} val samples from "
            f"{n_total - len(self._failed_users)} user/split combos "
            f"(rows available: {total_available_rows}, failed: {len(self._failed_users)})"
        )

        # Save to cache in chunks of 1000 to avoid WsFS 500MB file limit
        if cache_path:
            os.makedirs(cache_path, exist_ok=True)
            chunk_size = 1000
            n_chunks = (len(self._data_list) + chunk_size - 1) // chunk_size
            for i in range(n_chunks):
                chunk = self._data_list[i * chunk_size : (i + 1) * chunk_size]
                torch.save(chunk, os.path.join(cache_path, f"chunk_{i:03d}.pt"))
            print(f"  Saved val cache to: {cache_path} ({n_chunks} chunks)")

    @property
    def total_samples(self):
        return len(self._data_list)

    def __len__(self):
        return len(self._data_list) // self.batch_size

    def __iter__(self):
        for start in range(0, len(self._data_list) - self.batch_size + 1, self.batch_size):
            batch = Batch.from_data_list(self._data_list[start : start + self.batch_size])
            yield batch

    def get_label_counts(self):
        counts = {}
        for data in self._data_list:
            label = data.y.item()
            counts[label] = counts.get(label, 0) + 1
        return counts


In [ ]:
train_loader = ChunkedGNNTrainLoader(
    train_user_items, user_data, graph,
    batch_size=BATCH_SIZE, chunk_size=TRAIN_CHUNK_SIZE,
)

print(f"Train loader: {len(train_user_items)} users, "
      f"chunk_size={TRAIN_CHUNK_SIZE}, batch_size={BATCH_SIZE}")

# Compute class weights from train labels
print("\nComputing class weights from train labels...")
label_counts = train_loader.get_label_counts()
total_train = sum(label_counts.values())
classes = np.array(sorted(label_counts.keys()))
class_weights = torch.tensor(
    [total_train / (len(classes) * label_counts[c]) for c in classes],
    dtype=torch.float,
)
print(f"  Label counts: {label_counts}")
print(f"  Class weights: {class_weights.tolist()}")
print(f"  Total train samples (from y_tr): {total_train}")

In [ ]:
VAL_CACHE_PATH = os.path.join(EXPERIMENT_DIR, "val_samples_cache") if MAX_VAL_SAMPLES else None

val_loader = CachedValLoader(
    val_user_splits, user_data, graph,
    batch_size=BATCH_SIZE, max_samples=MAX_VAL_SAMPLES,
    cache_path=VAL_CACHE_PATH,
)
print(f"\nVal loader: {val_loader.total_samples} cached samples, {len(val_loader)} batches")

In [ ]:
# Anti-overfitting: strong regularization to prevent memorizing train users' graph patterns.
# Test set has entirely unseen users — model must generalize graph structure, not memorize it.
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=0.1,           # high dropout forces redundant representations that transfer
    drop_edge_rate=0.1,    # drop edges — prevents memorizing specific neighbor patterns
).to(device)

# Only apply INIT_WEIGHTS_PATH when starting fresh (no existing checkpoint from this experiment).
# This prevents accidentally re-initializing from pre-trained weights when resuming a run.
_existing_checkpoint = os.path.join(EXPERIMENT_DIR, "best_retweet_gnn_general.pt")
_is_fresh_start = RESET_GNN_TRAINING or not os.path.exists(_existing_checkpoint)
_apply_init_weights = bool(INIT_WEIGHTS_PATH) and _is_fresh_start

if not _is_fresh_start and INIT_WEIGHTS_PATH:
    print(f"⚠️  Existing checkpoint found at {_existing_checkpoint} — skipping INIT_WEIGHTS_PATH.")
    print(f"   (Set RESET_GNN_TRAINING=True to force re-initialization from pre-trained weights.)")

if _apply_init_weights:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
    state_dict = torch.load(INIT_WEIGHTS_PATH, map_location=device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Missing keys (will use random init): {missing}")
    if unexpected:
        print(f"  Unexpected keys (ignored): {unexpected}")
    print(f"  Weights loaded successfully. gate_param = {model.gate_param.item():.4f}")
else:
    # Gate at 0.0 → sigmoid=0.5 (balanced start). Let the model earn GNN contribution.
    # Shortcut head uses aggregate stats (rt_frac, node_count) which generalize better to unseen users.
    # If GNN can't beat shortcut on val, gate will stay low — that's fine.
    with torch.no_grad():
        model.gate_param.fill_(0.0)

In [ ]:
# user_data and graph are kept alive — needed by ChunkedGNNTrainLoader each epoch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Train with on-the-fly GNN DataLoaders — anti-overfitting configuration:
#   - lr=3e-3 with 5-epoch linear warmup: prevents fast memorization in early steps
#   - weight_decay=1e-3: L2 prevents weight specialization
#   - dropout=0.5 + drop_edge=0.1: stochastic regularization
#   - gate_param=0.0 (50/50): shortcut generalizes better; GNN must earn its contribution
#   - ChunkedGNNTrainLoader shuffles user order + within-chunk each epoch

import sys
from datetime import datetime
from contextlib import contextmanager

class TeeLogger:
    """Tee stdout to both the original stream and a timestamped log file."""
    def __init__(self, log_path, original_stdout):
        self._original = original_stdout
        self._file = open(log_path, "a", buffering=1)  # line-buffered
        self._line_buffer = ""

    def write(self, msg):
        self._original.write(msg)
        # Add timestamp at the start of each complete line
        for char in msg:
            if char == "\n":
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self._file.write(f"[{timestamp}] {self._line_buffer}\n")
                self._line_buffer = ""
            else:
                self._line_buffer += char

    def flush(self):
        self._original.flush()
        self._file.flush()

    def close(self):
        # Flush any remaining buffer
        if self._line_buffer:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            self._file.write(f"[{timestamp}] {self._line_buffer}\n")
            self._line_buffer = ""
        self._file.close()

@contextmanager
def tee_to_log(log_path):
    """Context manager that tees all stdout to a timestamped log file."""
    original_stdout = sys.stdout
    tee = TeeLogger(log_path, original_stdout)
    sys.stdout = tee
    try:
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()
        print(f"Training log saved to: {log_path}")

TRAINING_LOG_PATH = f"{EXPERIMENT_DIR}/training.log"

# Reduce LR and weight decay when fine-tuning from pre-trained weights
# (only applies when init weights were actually loaded — i.e. fresh start)
_base_lr = 3e-3
_base_wd = 1e-3
_train_lr = _base_lr * FINETUNE_LR_FACTOR if _apply_init_weights else _base_lr
_train_wd = _base_wd * FINETUNE_WD_FACTOR if _apply_init_weights else _base_wd
_warmup_epochs = FINETUNE_WARMUP_EPOCHS if _apply_init_weights else 5
if _apply_init_weights:
    print(f"Fine-tuning mode:")
    print(f"  LR reduced from {_base_lr:.1e} to {_train_lr:.1e} (factor={FINETUNE_LR_FACTOR})")
    print(f"  Weight decay reduced from {_base_wd:.1e} to {_train_wd:.1e} (factor={FINETUNE_WD_FACTOR})")
    print(f"  Warmup epochs reduced from 5 to {_warmup_epochs}")

with tee_to_log(TRAINING_LOG_PATH):
    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        experiment_dir=EXPERIMENT_DIR,
        class_weights=class_weights,
        epochs=EPOCHS,
        device=device,
        lr=_train_lr,
        log_every_n_steps=LOG_EVERY_N_STEPS,
        patience=PATIENCE,
        lr_warmup_epochs=_warmup_epochs,
        weight_decay=_train_wd,
        resume=False,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=MIXED_PRECISION,
        train_f1_every_n_epochs=TRAIN_F1_EVERY_N_EPOCHS,
    )